# DL_01: Data Preparation & Preprocessing Pipeline
**Causal NLP Reasoning — Phase 2: Deep Learning**

---

## 1. Purpose and Scope
This notebook establishes reproducible data preparation and preprocessing pipelines for both Phase 2 benchmark streams:
1. **Corr2Cause**: Abstract causal inference formulated as Natural Language Inference (NLI) premise-hypothesis entailment pairs.
2. **BioCause**: Candidate-conditioned directed causal relation classification using annotated argument spans from open-access PMC biomedical articles.

Key requirements:
- Prepare Corr2Cause and BioCause datasets independently (strict boundary enforcement).
- Document dataset schemas, label distributions, and directional label semantics.
- Ingest canonical input files from `data/raw/` and `data/processed/`.
- Preserve the canonical Corr2Cause split counts: Train (205,734), Validation (1,076), and Test (1,162).
- Maintain the 12-train / 4-validation / 4-test PMC article-level split isolation for BioCause to prevent cross-article data leakage.
- Construct explicit argument boundary markers (`[E1]...[/E1]` and `[E2]...[/E2]`) preserving original argument order and contextual text.
- Define expected model input tensors and target outputs for transformer fine-tuning in `DL_02` and `DL_03`.


## 2. Experimental Methodology & Theoretical Background

### Causal Relation Classification vs. Causal Entailment
- **Corr2Cause (Symbolic NLI)**: Evaluates whether language models deduce valid causal graph relations solely from statistical independence statements:
  - **Premise**: Verbalized statistical independence constraints among variables.
  - **Hypothesis**: Directed causal conjecture (e.g. "*A causes B.*")
  - **Target Label**: `1` (Entailed) or `0` (Refuted).
  - **Canonical Split Distribution**: Train = 205,734; Validation = 1,076; Test = 1,162. Validation must remain strictly separate from training.

- **BioCause (Biomedical Candidate-Conditioned Relation Classification)**: Given co-occurring biomedical entity spans ($Arg_1$, $Arg_2$) in scientific text, classify whether the directed relationship $Arg_1 \to Arg_2$ holds:
  - **Label Semantics**: Label `1` denotes that $Arg_1$ causes $Arg_2$. Label `0` denotes that $Arg_1$ does not represent the gold directed relation $Arg_1 \to Arg_2$ under the candidate-generation convention (which includes reversed directional pairs $(Arg_2, Arg_1)$ as well as non-causal pairs). Label `0` must not be mischaracterized as universally non-causal.
  - **Class Imbalance**: Candidate pairing produces 10,086 training pairs (621 positive, 9,465 negative), an actual negative-to-positive ratio of **15.24:1** (Validation: 131 pos / 1,783 neg = 13.61:1; Test: 100 pos / 1,308 neg = 13.08:1).

### Article-Level Split Isolation (Zero Data Leakage)
BioCause strictly partitions the 20 PMC publications by article ID:
- **Train Split**: 12 PMC Articles (621 gold causal relations, 10,086 candidate pairs)
- **Validation Split**: 4 PMC Articles (131 gold causal relations, 1,914 candidate pairs)
- **Test Split**: 4 PMC Articles (100 gold causal relations, 1,408 candidate pairs)


## 3. Expected Inputs

| Stream | File Path | Canonical Size / Dimensions | Key Fields |
| :--- | :--- | :--- | :--- |
| **Corr2Cause Train** | `data/raw/train.csv` | 205,734 rows | `input, label, num_variables, template` |
| **Corr2Cause Val** | `data/raw/validation.csv` | 1,076 rows | `input, label, num_variables, template` |
| **Corr2Cause Test** | `data/raw/test.csv` | 1,162 rows | `input, label, num_variables, template` |
| **BioCause Candidates** | `data/processed/biocause_candidates.csv` | 13,408 candidate pairs | `candidate_id, article_id, section_id, arg1_text, arg2_text, between_text, label, split...` |
| **BioCause Gold Relations** | `data/processed/biocause_relations.csv` | 852 gold relations | `relation_id, article_id, cause_id, effect_id...` |


## 4. Technical Responsibilities & Implementation Requirements

1. **Environment & Portable Root Discovery**: Locate repo root dynamically and load `dl_utils`.
2. **Corr2Cause Split Ingestion**: Verify exact row counts (205,734 train, 1,076 val, 1,162 test) and verify binary label balance.
3. **BioCause Article-Level Isolation Audit**: Check that article IDs across splits are strictly disjoint.
4. **Argument Marker Construction**: Format BioCause inputs with special tokens `[E1]`, `[/E1]`, `[E2]`, `[/E2]`. Ensure markers reference original text offsets, preserve original text order, include surrounding context, and check for span validity/overlap.
5. **Dynamic Class Weight Calculation**: Compute $w_{\text{pos}} = N_{\text{neg}} / N_{\text{pos}}$ strictly on the training split ($9,465 / 621 \approx 15.24$). Never leak validation or test counts into weight computation.
6. **Token Length Profiling**: Profile sequence lengths under `distilbert-base-uncased` and `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` tokenizers.
7. **Dataset Serialization**: Export clean model-ready splits to `data/processed/`.


In [ ]:
# Section 1: Environment Setup & Portable Imports
import sys
import os
from pathlib import Path

def find_repo_root():
    curr = Path(os.getcwd()).resolve()
    for p in [curr] + list(curr.parents):
        if (p / 'AML' / 'causal_extractors.py').exists() and (p / 'DL' / 'dl_utils.py').exists():
            return p
    return curr

ROOT_DIR = find_repo_root()
for p in [ROOT_DIR, ROOT_DIR / 'AML', ROOT_DIR / 'DL']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import dl_utils
print(f'Repository root located at: {ROOT_DIR}')


In [ ]:
# Section 2: Corr2Cause Data Ingestion & Split Verification
import pandas as pd
import numpy as np

# TODO: Load canonical Corr2Cause files and verify exact row counts:
# Expected: Train = 205,734 | Validation = 1,076 | Test = 1,162
CORR2CAUSE_RAW_DIR = ROOT_DIR / 'data' / 'raw'
# df_c2c_train = pd.read_csv(CORR2CAUSE_RAW_DIR / 'train.csv')
# df_c2c_val = pd.read_csv(CORR2CAUSE_RAW_DIR / 'validation.csv')
# df_c2c_test = pd.read_csv(CORR2CAUSE_RAW_DIR / 'test.csv')
# assert len(df_c2c_train) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['train']
# assert len(df_c2c_val) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['validation']
# assert len(df_c2c_test) == dl_utils.CORR2CAUSE_SPLIT_COUNTS['test']


In [ ]:
# Section 3: BioCause Data Ingestion & Article-Level Split Isolation Audit
# TODO: Load biocause_candidates.csv and verify article-level partition
# Expected splits: 12 train articles, 4 validation articles, 4 test articles
BIOCAUSE_CANDS_PATH = ROOT_DIR / 'data' / 'processed' / 'biocause_candidates.csv'
# df_bio_cands = pd.read_csv(BIOCAUSE_CANDS_PATH)
# train_articles = set(df_bio_cands[df_bio_cands['split'] == 'train']['article_id'])
# val_articles = set(df_bio_cands[df_bio_cands['split'] == 'validation']['article_id'])
# test_articles = set(df_bio_cands[df_bio_cands['split'] == 'test']['article_id'])
# assert len(train_articles & val_articles) == 0
# assert len(train_articles & test_articles) == 0
# assert len(val_articles & test_articles) == 0


In [ ]:
# Section 4: Dynamic Training Class Weight Calculation
# TODO: Compute positive class weight strictly from training split (621 pos, 9,465 neg)
# pos_weight = dl_utils.calculate_dynamic_class_weight(n_negative=9465, n_positive=621)
# Expected pos_weight approx 15.2415


In [ ]:
# Section 5: Argument Boundary Marker Formatting
# TODO: Implement formatting function inserting [E1]...[/E1] and [E2]...[/E2] into source context
# - Preserves original text order of arguments
# - Validates offsets against source text
# - Rejects or reports overlapping spans
# - Includes full sentence/section context, not merely between_text


In [ ]:
# Section 6: Sequence Length Distribution & Tokenizer Profiling
# TODO: Profile token length distribution with:
# - DistilBERT tokenizer: dl_utils.DISTILBERT_CHECKPOINT
# - PubMedBERT tokenizer: dl_utils.PUBMEDBERT_CHECKPOINT
# Determine optimal max_length truncation boundaries without truncating marked entities


In [ ]:
# Section 7: Export Model-Ready Datasets
# TODO: Export verified preprocessed datasets to data/processed/:
# - dl_corr2cause_train.csv, dl_corr2cause_val.csv, dl_corr2cause_test.csv
# - dl_biocause_train.csv, dl_biocause_val.csv, dl_biocause_test.csv


## 5. Expected Outputs & Evaluation Criteria

- `data/processed/dl_corr2cause_{train,val,test}.csv`: Cleaned NLI splits conforming to canonical split counts.
- `data/processed/dl_biocause_{train,val,test}.csv`: Marked relation candidate datasets with preserved 12/4/4 article grouping.
- **Evaluation Criteria**:
  - 100% split integrity: Zero overlap between article IDs in train, val, and test.
  - Argument marker completeness: Both entity spans intact post-formatting and tokenization.
  - Class weight verification: Dynamic training ratio $w_{\text{pos}} \approx 15.24$ computed strictly from training data.
